In [17]:
import torch
import torch.nn as nn
import torchaudio
import soundfile as sf
import numpy as np
import json
from pathlib import Path

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device : {DEVICE}")
print(f"   torch  : {torch.__version__}")

✅ Device : cpu
   torch  : 2.10.0+cpu


In [18]:
BASE_DIR       = Path("../")
PROCESSED_DIR  = BASE_DIR / "data/processed"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"

VOCAB_PATH     = PROCESSED_DIR / "vocab.json"
MODEL_PATH     = CHECKPOINT_DIR / "best_model.pt"

print(f"Vocab exists  : {VOCAB_PATH.exists()}")
print(f"Model exists  : {MODEL_PATH.exists()}")

Vocab exists  : True
Model exists  : True


In [19]:
with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    char2idx = json.load(f)

idx2char   = {i: ch for ch, i in char2idx.items()}
VOCAB_SIZE = len(char2idx)

print(f"✅ Vocab size : {VOCAB_SIZE}")
print(f"   Sample     : {list(char2idx.items())[:5]}")

✅ Vocab size : 41
   Sample     : [('<blank>', 0), (' ', 1), ('ئ', 2), ('ا', 3), ('ب', 4)]


In [20]:
class UrduASRModel(nn.Module):
    def __init__(self, n_mels=80, vocab_size=VOCAB_SIZE,
                 rnn_hidden=256, rnn_layers=3, dropout=0.3):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(dropout)
        )

        cnn_out_size = (n_mels // 2) * 32

        self.rnn = nn.GRU(
            input_size=cnn_out_size,
            hidden_size=rnn_hidden,
            num_layers=rnn_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.fc = nn.Linear(rnn_hidden * 2, vocab_size)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.cnn(x)
        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(b, t, c * f)
        x, _ = self.rnn(x)
        x = self.fc(x)
        x = x.permute(1, 0, 2)
        return x

print("✅ Model architecture defined")

✅ Model architecture defined


In [21]:
model = UrduASRModel(vocab_size=VOCAB_SIZE).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("✅ Model loaded successfully")
print(f"   Parameters : {sum(p.numel() for p in model.parameters()):,}")

✅ Model loaded successfully
   Parameters : 4,758,537


In [22]:
def extract_mel(wav_path, sample_rate=16000, n_mels=80, n_fft=400, hop_length=160, max_seconds=3):
    audio_data, sr = sf.read(wav_path)
    
    # Convert stereo to mono if needed
    if len(audio_data.shape) > 1:
        audio_data = audio_data.mean(axis=1)
    
    # Trim or pad to max_seconds
    max_samples = sample_rate * max_seconds
    if len(audio_data) > max_samples:
        audio_data = audio_data[:max_samples]
    elif len(audio_data) < n_fft:
        # Pad if too short
        audio_data = np.pad(audio_data, (0, n_fft - len(audio_data)))

    waveform = torch.tensor(audio_data, dtype=torch.float32).unsqueeze(0)
    
    if sr != sample_rate:
        waveform = torchaudio.transforms.Resample(sr, sample_rate)(waveform)
    
    mel = torchaudio.transforms.MelSpectrogram(
        sample_rate=sample_rate, n_fft=n_fft,
        hop_length=hop_length, n_mels=n_mels
    )(waveform)
    mel_db = torchaudio.transforms.AmplitudeToDB()(mel)
    return mel_db.squeeze(0)  # (n_mels, time)

print("✅ Decoder ready")

✅ Decoder ready


In [23]:
def predict(wav_path):
    """
    Full pipeline:
    WAV file → Mel Spectrogram → Model → Greedy Decode → Urdu Text
    """
    # Extract features
    feature = extract_mel(wav_path)                         # (n_mels, time)
    feature = feature.unsqueeze(0).to(DEVICE)               # (1, n_mels, time)

    # Run through model
    with torch.no_grad():
        output = model(feature)                             # (time, 1, vocab_size)
        output = output.squeeze(1)                          # (time, vocab_size)
        probs  = torch.nn.functional.softmax(output, dim=-1)

    # Decode
    predicted_text = greedy_decode(probs, idx2char)

    return predicted_text


print("✅ Predict function ready")

✅ Predict function ready


In [24]:
import pandas as pd

df = pd.read_csv(PROCESSED_DIR / "metadata.csv")

# Test on 5 random samples
samples = df.sample(5, random_state=42)

print("=" * 50)
print(f"{'FILE':<20} {'ACTUAL':<15} {'PREDICTED':<15}")
print("=" * 50)

for _, row in samples.iterrows():
    predicted = predict(row['file_path'])
    filename  = Path(row['file_path']).stem
    print(f"{filename:<20} {row['word']:<15} {predicted:<15}")

FILE                 ACTUAL          PREDICTED      
AGMNG1198            دتشہ            دتشہ           
AFFYG1115            ضعب             ضعب            
AFFYG1065            دصر             دصر            
AKMNG2038            دربمس           دربمس          
AHMNG1038            دربمس           دربمس          


In [ ]:
# Put any urdu WAV file in your project and update the path
MY_WAV = "../data/raw/archive/files/AAMNG1/AAMNG1068.wav"

predicted = predict(MY_WAV)
print(f"🎙️  Predicted Urdu text : {predicted}")

🎙️  Predicted Urdu text : رفص
